### Imports

In [2]:
import pandas as pd

### Chargement des fichiers

In [3]:
application = pd.read_csv("../data/application_train.csv")
bureau = pd.read_csv("../data/bureau.csv")
previous_application = pd.read_csv("../data/previous_application.csv")
pos = pd.read_csv("../data/POS_CASH_balance.csv")
installments = pd.read_csv("../data/installments_payments.csv")
credit = pd.read_csv("../data/credit_card_balance.csv")
bureau_balance = pd.read_csv("../data/bureau_balance.csv")

In [4]:
# Vérification de la table principale
print(application.shape)
application.head()

(307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
# Copie de la table principale à enrichir
df = application.copy()

### Préparation table enrichie

#### bureau_balance

In [6]:
# Agrégation de bureau_balance au niveau du crédit (SK_ID_BUREAU)
bb_num_agg = bureau_balance.groupby("SK_ID_BUREAU").agg({
    "MONTHS_BALANCE": ["min", "max", "size"]
})

bb_num_agg.columns = ["BB_" + "_".join(col) for col in bb_num_agg.columns]

In [7]:
# Encodage de STATUS puis agrégation au niveau du crédit
bb_status = pd.get_dummies(bureau_balance["STATUS"], prefix="BB_STATUS")
bb_status["SK_ID_BUREAU"] = bureau_balance["SK_ID_BUREAU"]

bb_status_agg = bb_status.groupby("SK_ID_BUREAU").mean()

In [8]:
# Fusion des agrégations bureau_balance
bb_agg = bb_num_agg.join(bb_status_agg, how="left").reset_index()
print(bb_agg.shape)
bb_agg.head()

(817395, 12)


,SK_ID_BUREAU,BB_MONTHS_BALANCE_min,BB_MONTHS_BALANCE_max,BB_MONTHS_BALANCE_size,BB_STATUS_0,BB_STATUS_1,BB_STATUS_2,BB_STATUS_3,BB_STATUS_4,BB_STATUS_5,BB_STATUS_C,BB_STATUS_X
0,5001709,-96,0,97,0.000000,0.0,0.0,0.0,0.0,0.0,0.886598,0.113402
1,5001710,-82,0,83,0.060241,0.0,0.0,0.0,0.0,0.0,0.578313,0.361446
2,5001711,-3,0,4,0.750000,0.0,0.0,0.0,0.0,0.0,0.000000,0.250000
3,5001712,-18,0,19,0.526316,0.0,0.0,0.0,0.0,0.0,0.473684,0.000000
4,5001713,-21,0,22,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,1.000000


In [9]:
# Enrichissement de bureau avec les informations issues de bureau_balance
bureau = bureau.merge(bb_agg, on="SK_ID_BUREAU", how="left")
print(bureau.shape)

(1716428, 28)


In [10]:
# Agrégation de bureau au niveau client (SK_ID_CURR)
bureau_agg = bureau.groupby("SK_ID_CURR").agg({
    "DAYS_CREDIT": ["mean", "max"],
    "AMT_CREDIT_SUM": ["mean", "sum"],
    "AMT_CREDIT_SUM_DEBT": ["mean", "sum"],
    "BB_MONTHS_BALANCE_min": ["mean"],
    "BB_MONTHS_BALANCE_max": ["mean"],
    "BB_MONTHS_BALANCE_size": ["mean", "sum"],
    "BB_STATUS_0": ["mean"],
    "BB_STATUS_1": ["mean"],
    "BB_STATUS_2": ["mean"],
    "BB_STATUS_3": ["mean"],
    "BB_STATUS_4": ["mean"],
    "BB_STATUS_5": ["mean"],
    "BB_STATUS_C": ["mean"],
    "BB_STATUS_X": ["mean"]
})

bureau_agg.columns = ["BUREAU_" + "_".join(col) for col in bureau_agg.columns]
bureau_agg = bureau_agg.reset_index()

print(bureau_agg.shape)
bureau_agg.head()

(305811, 19)


,SK_ID_CURR,BUREAU_DAYS_CREDIT_mean,BUREAU_DAYS_CREDIT_max,BUREAU_AMT_CREDIT_SUM_mean,BUREAU_AMT_CREDIT_SUM_sum,BUREAU_AMT_CREDIT_SUM_DEBT_mean,BUREAU_AMT_CREDIT_SUM_DEBT_sum,BUREAU_BB_MONTHS_BALANCE_min_mean,BUREAU_BB_MONTHS_BALANCE_max_mean,BUREAU_BB_MONTHS_BALANCE_size_mean,BUREAU_BB_MONTHS_BALANCE_size_sum,BUREAU_BB_STATUS_0_mean,BUREAU_BB_STATUS_1_mean,BUREAU_BB_STATUS_2_mean,BUREAU_BB_STATUS_3_mean,BUREAU_BB_STATUS_4_mean,BUREAU_BB_STATUS_5_mean,BUREAU_BB_STATUS_C_mean,BUREAU_BB_STATUS_X_mean
0,100001,-735.000000,-49,207623.571429,1453365.000,85240.928571,596686.5,-23.571429,0.0,24.571429,172.0,0.336651,0.007519,0.0,0.0,0.0,0.0,0.441240,0.214590
1,100002,-874.000000,-103,108131.945625,865055.565,49156.200000,245781.0,-28.250000,-15.5,13.750000,110.0,0.406960,0.255682,0.0,0.0,0.0,0.0,0.175426,0.161932
2,100003,-1400.750000,-606,254350.125000,1017400.500,0.000000,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100004,-867.000000,-408,94518.900000,189037.800,0.000000,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100005,-190.666667,-62,219042.000000,657126.000,189469.500000,568408.5,-6.000000,0.0,7.000000,21.0,0.735043,0.000000,0.0,0.0,0.0,0.0,0.128205,0.136752


In [11]:
# Fusion avec la table principale
df = df.merge(bureau_agg, on="SK_ID_CURR", how="left")
print(df.shape)

(307511, 140)


#### previous_application

In [12]:
# Agrégation de previous_application au niveau client
previous_agg = previous_application.groupby("SK_ID_CURR").agg({
    "AMT_APPLICATION": ["mean", "max"],
    "AMT_CREDIT": ["mean", "max"],
    "AMT_ANNUITY": ["mean"],
    "CNT_PAYMENT": ["mean", "sum"],
    "DAYS_DECISION": ["mean", "max"]
})

previous_agg.columns = ["PREV_" + "_".join(col) for col in previous_agg.columns]
previous_agg = previous_agg.reset_index()

print(previous_agg.shape)
previous_agg.head()

(338857, 10)


,SK_ID_CURR,PREV_AMT_APPLICATION_mean,PREV_AMT_APPLICATION_max,PREV_AMT_CREDIT_mean,PREV_AMT_CREDIT_max,PREV_AMT_ANNUITY_mean,PREV_CNT_PAYMENT_mean,PREV_CNT_PAYMENT_sum,PREV_DAYS_DECISION_mean,PREV_DAYS_DECISION_max
0,100001,24835.50,24835.5,23787.00,23787.0,3951.000,8.0,8.0,-1740.0,-1740
1,100002,179055.00,179055.0,179055.00,179055.0,9251.775,24.0,24.0,-606.0,-606
2,100003,435436.50,900000.0,484191.00,1035882.0,56553.990,10.0,30.0,-1305.0,-746
3,100004,24282.00,24282.0,20106.00,20106.0,5357.250,4.0,4.0,-815.0,-815
4,100005,22308.75,44617.5,20076.75,40153.5,4813.200,12.0,12.0,-536.0,-315


In [13]:
# Fusion avec la table principale
df = df.merge(previous_agg, on="SK_ID_CURR", how="left")
print(df.shape)

(307511, 149)


#### POS_CASH_balance

In [14]:
# Agrégation de POS_CASH_balance au niveau client
pos_agg = pos.groupby("SK_ID_CURR").agg({
    "MONTHS_BALANCE": ["min", "max"],
    "SK_DPD": ["mean", "max"],
    "SK_DPD_DEF": ["mean", "max"],
    "CNT_INSTALMENT": ["mean", "sum"],
    "CNT_INSTALMENT_FUTURE": ["mean", "sum"]
})

pos_agg.columns = ["POS_" + "_".join(col) for col in pos_agg.columns]
pos_agg = pos_agg.reset_index()

print(pos_agg.shape)
pos_agg.head()

(337252, 11)


,SK_ID_CURR,POS_MONTHS_BALANCE_min,POS_MONTHS_BALANCE_max,POS_SK_DPD_mean,POS_SK_DPD_max,POS_SK_DPD_DEF_mean,POS_SK_DPD_DEF_max,POS_CNT_INSTALMENT_mean,POS_CNT_INSTALMENT_sum,POS_CNT_INSTALMENT_FUTURE_mean,POS_CNT_INSTALMENT_FUTURE_sum
0,100001,-96,-53,0.777778,7,0.777778,7,4.000000,36.0,1.444444,13.0
1,100002,-19,-1,0.000000,0,0.000000,0,24.000000,456.0,15.000000,285.0
2,100003,-77,-18,0.000000,0,0.000000,0,10.107143,283.0,5.785714,162.0
3,100004,-27,-24,0.000000,0,0.000000,0,3.750000,15.0,2.250000,9.0
4,100005,-25,-15,0.000000,0,0.000000,0,11.700000,117.0,7.200000,72.0


In [15]:
# Fusion avec la table principale
df = df.merge(pos_agg, on="SK_ID_CURR", how="left")
print(df.shape)

(307511, 159)


#### installments_payments

In [16]:
# Agrégation de installments_payments au niveau client
installments_agg = installments.groupby("SK_ID_CURR").agg({
    "NUM_INSTALMENT_VERSION": ["nunique"],
    "AMT_INSTALMENT": ["mean", "sum", "max"],
    "AMT_PAYMENT": ["mean", "sum", "max"],
    "DAYS_ENTRY_PAYMENT": ["mean", "max"],
    "DAYS_INSTALMENT": ["mean", "max"]
})

installments_agg.columns = ["INST_" + "_".join(col) for col in installments_agg.columns]
installments_agg = installments_agg.reset_index()

print(installments_agg.shape)
installments_agg.head()

(339587, 12)


,SK_ID_CURR,INST_NUM_INSTALMENT_VERSION_nunique,INST_AMT_INSTALMENT_mean,INST_AMT_INSTALMENT_sum,INST_AMT_INSTALMENT_max,INST_AMT_PAYMENT_mean,INST_AMT_PAYMENT_sum,INST_AMT_PAYMENT_max,INST_DAYS_ENTRY_PAYMENT_mean,INST_DAYS_ENTRY_PAYMENT_max,INST_DAYS_INSTALMENT_mean,INST_DAYS_INSTALMENT_max
0,100001,2,5885.132143,41195.925,17397.900,5885.132143,41195.925,17397.900,-2195.000000,-1628.0,-2187.714286,-1619.0
1,100002,2,11559.247105,219625.695,53093.745,11559.247105,219625.695,53093.745,-315.421053,-49.0,-295.000000,-25.0
2,100003,2,64754.586000,1618864.650,560835.360,64754.586000,1618864.650,560835.360,-1385.320000,-544.0,-1378.160000,-536.0
3,100004,2,7096.155000,21288.465,10573.965,7096.155000,21288.465,10573.965,-761.666667,-727.0,-754.000000,-724.0
4,100005,2,6240.205000,56161.845,17656.245,6240.205000,56161.845,17656.245,-609.555556,-470.0,-586.000000,-466.0


In [17]:
# Fusion avec la table principale
df = df.merge(installments_agg, on="SK_ID_CURR", how="left")
print(df.shape)

(307511, 170)


#### credit_card_balance

In [18]:
# Agrégation de credit_card_balance au niveau client
credit_agg = credit.groupby("SK_ID_CURR").agg({
    "MONTHS_BALANCE": ["min", "max"],
    "AMT_BALANCE": ["mean", "max"],
    "AMT_CREDIT_LIMIT_ACTUAL": ["mean", "max"],
    "AMT_DRAWINGS_CURRENT": ["mean", "sum"],
    "CNT_DRAWINGS_CURRENT": ["mean", "sum"],
    "SK_DPD": ["mean", "max"],
    "SK_DPD_DEF": ["mean", "max"]
})

credit_agg.columns = ["CC_" + "_".join(col) for col in credit_agg.columns]
credit_agg = credit_agg.reset_index()

print(credit_agg.shape)
credit_agg.head()

(103558, 15)


,SK_ID_CURR,CC_MONTHS_BALANCE_min,CC_MONTHS_BALANCE_max,CC_AMT_BALANCE_mean,CC_AMT_BALANCE_max,CC_AMT_CREDIT_LIMIT_ACTUAL_mean,CC_AMT_CREDIT_LIMIT_ACTUAL_max,CC_AMT_DRAWINGS_CURRENT_mean,CC_AMT_DRAWINGS_CURRENT_sum,CC_CNT_DRAWINGS_CURRENT_mean,CC_CNT_DRAWINGS_CURRENT_sum,CC_SK_DPD_mean,CC_SK_DPD_max,CC_SK_DPD_DEF_mean,CC_SK_DPD_DEF_max
0,100006,-6,-1,0.000000,0.00,270000.000000,270000,0.000000,0.0,0.000000,0,0.000000,0,0.000000,0
1,100011,-75,-2,54482.111149,189000.00,164189.189189,180000,2432.432432,180000.0,0.054054,4,0.000000,0,0.000000,0
2,100013,-96,-1,18159.919219,161420.22,131718.750000,157500,5953.125000,571500.0,0.239583,23,0.010417,1,0.010417,1
3,100021,-18,-2,0.000000,0.00,675000.000000,675000,0.000000,0.0,0.000000,0,0.000000,0,0.000000,0
4,100023,-11,-4,0.000000,0.00,135000.000000,225000,0.000000,0.0,0.000000,0,0.000000,0,0.000000,0


In [19]:
# Fusion avec la table principale
df = df.merge(credit_agg, on="SK_ID_CURR", how="left")
print(df.shape)

(307511, 184)


### Vérification finale

In [20]:
# Vérification de la cohérence du dataset final
print("Shape finale :", df.shape)
print("Nombre de clients uniques :", df["SK_ID_CURR"].nunique())
print("Doublons SK_ID_CURR :", df.duplicated("SK_ID_CURR").sum())

Shape finale : (307511, 184)
Nombre de clients uniques : 307511
Doublons SK_ID_CURR : 0


In [21]:
# Aperçu final
df.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,CC_AMT_CREDIT_LIMIT_ACTUAL_mean,CC_AMT_CREDIT_LIMIT_ACTUAL_max,CC_AMT_DRAWINGS_CURRENT_mean,CC_AMT_DRAWINGS_CURRENT_sum,CC_CNT_DRAWINGS_CURRENT_mean,CC_CNT_DRAWINGS_CURRENT_sum,CC_SK_DPD_mean,CC_SK_DPD_max,CC_SK_DPD_DEF_mean,CC_SK_DPD_DEF_max
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,270000.0,270000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
# Colonnes les plus incomplètes
df.isnull().mean().sort_values(ascending=False).head(30)

CC_SK_DPD_DEF_mean                    0.717392
CC_SK_DPD_DEF_max                     0.717392
CC_CNT_DRAWINGS_CURRENT_mean          0.717392
CC_CNT_DRAWINGS_CURRENT_sum           0.717392
CC_AMT_DRAWINGS_CURRENT_sum           0.717392
CC_AMT_DRAWINGS_CURRENT_mean          0.717392
CC_AMT_BALANCE_max                    0.717392
CC_AMT_BALANCE_mean                   0.717392
CC_AMT_CREDIT_LIMIT_ACTUAL_mean       0.717392
CC_AMT_CREDIT_LIMIT_ACTUAL_max        0.717392
CC_SK_DPD_mean                        0.717392
CC_SK_DPD_max                         0.717392
CC_MONTHS_BALANCE_max                 0.717392
CC_MONTHS_BALANCE_min                 0.717392
BUREAU_BB_STATUS_0_mean               0.700073
BUREAU_BB_STATUS_3_mean               0.700073
BUREAU_BB_STATUS_5_mean               0.700073
BUREAU_BB_STATUS_C_mean               0.700073
BUREAU_BB_STATUS_1_mean               0.700073
BUREAU_BB_STATUS_2_mean               0.700073
BUREAU_BB_MONTHS_BALANCE_max_mean     0.700073
BUREAU_BB_MON

In [23]:
# Sauvegarde du dataset
df.to_csv("../data/application_merged.csv", index=False)

### Conclusion

La table principale `application_train` a été enrichie à partir de plusieurs sources de données externes et historiques :

- `bureau` et `bureau_balance`
- `previous_application`
- `POS_CASH_balance`
- `installments_payments`
- `credit_card_balance`

Les tables secondaires ont été agrégées au niveau client (`SK_ID_CURR`) afin de conserver une structure finale d’une ligne par client.

Le dataset obtenu constitue une base fusionnée brute, qui sera analysée dans le notebook d’analyse exploratoire puis enrichie davantage lors de la phase de feature engineering.